# Milestone 1: ADM2 + WorldPop Under-18 Population Fusion

This notebook creates the first fused spatial dataset for ChildReach. It aggregates WorldPop 2019 under-age-18 raster values into Mozambique ADM2 boundary polygons using zonal statistics.

The output is a district-level GeoPackage and CSV with:

- estimated under-18 population (`under18_sum`)
- the number of valid WorldPop raster cells used (`valid_cell_count`)
- a transparent raster-support flag (`worldpop_valid_cells`)


## 1. Load libraries and input paths

The ADM2 boundaries define the analysis units. The WorldPop raster provides gridded under-18 population estimates. Both sources were previously inspected for CRS, bounds, NoData handling, and value plausibility.


In [15]:
from pathlib import Path
import sys

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

import geopandas as gpd
import pandas as pd
from rasterstats import zonal_stats

from childreach.paths import (
    ADM2_BOUNDARIES_GEOJSON,
    DATA_PROCESSED,
    MOZ_ADM2_UNDER18_2019_CSV,
    MOZ_ADM2_UNDER18_2019_GPKG,
    WORLDPOP_UNDER18_TOTAL_2019_TIF,
)

boundaries_path = ADM2_BOUNDARIES_GEOJSON
raster_path = WORLDPOP_UNDER18_TOTAL_2019_TIF
processed_dir = DATA_PROCESSED
processed_dir.mkdir(parents=True, exist_ok=True)

gpkg_path = MOZ_ADM2_UNDER18_2019_GPKG
csv_path = MOZ_ADM2_UNDER18_2019_CSV


## 2. Load ADM2 boundaries

Expected result: 159 ADM2 district/district-like polygons in EPSG:4326.


In [16]:
boundaries = gpd.read_file(boundaries_path)

print(boundaries.shape)
print(boundaries.crs)
print(boundaries[["shapeName", "shapeID", "shapeType", "shapeGroup"]].head())


(159, 6)
EPSG:4326
      shapeName                  shapeID shapeType shapeGroup
0  Alto Molocue  85939544B55280380491119      ADM2        MOZ
1       Ancuabe  85939544B12924931865401      ADM2        MOZ
2       Angoche  85939544B28799715012212      ADM2        MOZ
3       Angonia  85939544B12573256040215      ADM2        MOZ
4        Balama  85939544B22682737370097      ADM2        MOZ


## 3. Run zonal statistics

For each ADM2 polygon, `zonal_stats` summarizes valid raster cells inside the polygon.

- `sum` becomes the estimated under-18 population total for the district.
- `count` is the number of valid raster cells contributing to that estimate.
- `nodata=-99999` prevents WorldPop missing-value cells from being treated as population values.


In [17]:
stats = zonal_stats(
    boundaries,
    raster_path,
    stats=["sum", "count"],
    nodata=-99999,
)

stats_df = pd.DataFrame(stats)

print(stats_df.head())
print(stats_df.shape)
print(stats_df.isna().sum())


    count            sum
0  363972  213755.062500
1   51808   92980.007812
2   81424  204043.000000
3  180952  263586.812500
4   45246  103711.765625
(159, 2)
count    0
sum      2
dtype: int64


## 4. Join statistics back to ADM2 boundaries

Keep missing population estimates as `NaN` instead of converting them to zero. A missing value means there were no valid WorldPop raster cells for that polygon, not that the true under-18 population is zero.


In [18]:
results = boundaries.copy()
results["under18_sum"] = stats_df["sum"]
results["valid_cell_count"] = stats_df["count"]
results["worldpop_valid_cells"] = results["valid_cell_count"] > 0

print(results[["shapeName", "under18_sum", "valid_cell_count", "worldpop_valid_cells"]].head())
print(results["worldpop_valid_cells"].value_counts(dropna=False))
print(results["under18_sum"].describe())


      shapeName    under18_sum  valid_cell_count  worldpop_valid_cells
0  Alto Molocue  213755.062500            363972                  True
1       Ancuabe   92980.007812             51808                  True
2       Angoche  204043.000000             81424                  True
3       Angonia  263586.812500            180952                  True
4        Balama  103711.765625             45246                  True
worldpop_valid_cells
True     157
False      2
Name: count, dtype: int64
count       157.000000
mean      97704.932878
std       80159.699375
min        1746.807617
25%       46787.503906
50%       78038.890625
75%      124583.578125
max      473215.625000
Name: under18_sum, dtype: float64


## 5. Diagnose missing raster support

Two island ADM2 features have no valid WorldPop cells. This diagnostic keeps that limitation visible for the paper, map legend, and future app.


In [19]:
missing = results.loc[
    ~results["worldpop_valid_cells"],
    ["shapeName", "shapeID", "shapeType", "valid_cell_count", "under18_sum"],
]

print(missing)
print(missing["valid_cell_count"].describe())


        shapeName                  shapeID shapeType  valid_cell_count  \
53     Ilha Licom  85939544B53859921366962      ADM2                 0   
54  Ilha Risunodo  85939544B49635827229265      ADM2                 0   

    under18_sum  
53          NaN  
54          NaN  
count    2.0
mean     0.0
std      0.0
min      0.0
25%      0.0
50%      0.0
75%      0.0
max      0.0
Name: valid_cell_count, dtype: float64


## 6. Check whether `all_touched=True` changes the missing-island diagnosis

This is a focused diagnostic, not the primary analysis setting. If the islands still have zero valid cells with `all_touched=True`, the issue is raster support/coverage rather than only the default cell-center rule.


In [20]:
stats_all_touched = zonal_stats(
    boundaries,
    raster_path,
    stats=["sum", "count"],
    nodata=-99999,
    all_touched=True,
)

stats_all_touched_df = pd.DataFrame(stats_all_touched)

comparison = boundaries[["shapeName", "shapeID"]].copy()
comparison["count_default"] = stats_df["count"]
comparison["sum_default"] = stats_df["sum"]
comparison["count_all_touched"] = stats_all_touched_df["count"]
comparison["sum_all_touched"] = stats_all_touched_df["sum"]

print(
    comparison.loc[
        comparison["shapeName"].isin(["Ilha Licom", "Ilha Risunodo"]),
        [
            "shapeName",
            "count_default",
            "sum_default",
            "count_all_touched",
            "sum_all_touched",
        ],
    ]
)


        shapeName  count_default  sum_default  count_all_touched  \
53     Ilha Licom              0          NaN                  0   
54  Ilha Risunodo              0          NaN                  0   

    sum_all_touched  
53              NaN  
54              NaN  


## 7. Sanity-check highest and lowest valid ADM2 totals

This is not a formal validation against census totals. It is a plausibility check: large urban/populous districts should generally appear near the high end, while sparse or small districts should appear near the low end.


In [21]:
valid_results = results.loc[
    results["worldpop_valid_cells"],
    ["shapeName", "under18_sum", "valid_cell_count"],
]

print("Highest under-18 estimates")
print(valid_results.sort_values("under18_sum", ascending=False).head(10))

print("\nLowest under-18 estimates")
print(valid_results.sort_values("under18_sum", ascending=True).head(10))


Highest under-18 estimates
             shapeName    under18_sum  valid_cell_count
26    Cidade Da Matola  473215.625000             46310
30    Cidade De Maputo  442010.875000             32065
31   Cidade De Nampula  425966.875000             30442
108            Milange  363712.437500            351980
25     Cidade Da Beira  291995.781250             31719
3              Angonia  263586.812500            180952
48               Gurue  251897.593750            310634
112             Mocuba  242756.859375            320779
118             Monapo  234931.093750            157711
121         Morrumbala  231907.734375            250850

Lowest under-18 estimates
         shapeName   under18_sum  valid_cell_count
60     Lago Niassa   1746.807617              1167
50             Ibo   5953.359375              1586
18         Chigubo  10106.416016             38619
92      Massangena  11658.722656             25467
103         Mecula  12019.120117             14341
16   Chicualacuala  1570

## 8. Save processed outputs

The GeoPackage preserves geometry for future mapping and spatial analysis. The CSV provides a lightweight table for inspection, reporting, and non-spatial analysis.


In [22]:
results.to_file(gpkg_path, layer="moz_adm2_under18_2019", driver="GPKG")
results.drop(columns="geometry").to_csv(csv_path, index=False)

print(gpkg_path)
print(gpkg_path.exists(), gpkg_path.stat().st_size)
print(csv_path)
print(csv_path.exists(), csv_path.stat().st_size)


C:\Users\Cameron\Desktop\github\child_reach_spacial_project\data\processed\moz_adm2_under18_2019.gpkg
True 9134080
C:\Users\Cameron\Desktop\github\child_reach_spacial_project\data\processed\moz_adm2_under18_2019.csv
True 10967


## 9. Reload outputs to verify reproducibility

A successful reload confirms the saved files can be reused by later notebooks, scripts, maps, reports, or an app prototype.


In [23]:
processed_gdf = gpd.read_file(gpkg_path, layer="moz_adm2_under18_2019")
processed_csv = pd.read_csv(csv_path)

print(processed_gdf.shape)
print(processed_csv.shape)
print(processed_gdf.crs)
print(processed_gdf[["shapeName", "under18_sum", "valid_cell_count", "worldpop_valid_cells"]].head())
print(processed_gdf["worldpop_valid_cells"].value_counts(dropna=False))


(159, 9)
(159, 8)
EPSG:4326
      shapeName    under18_sum  valid_cell_count  worldpop_valid_cells
0  Alto Molocue  213755.062500            363972                  True
1       Ancuabe   92980.007812             51808                  True
2       Angoche  204043.000000             81424                  True
3       Angonia  263586.812500            180952                  True
4        Balama  103711.765625             45246                  True
worldpop_valid_cells
True     157
False      2
Name: count, dtype: int64


## Milestone 1 interpretation

This notebook completes the first ChildReach spatial fusion milestone: ADM2 boundaries plus WorldPop under-18 raster data have been combined into a district-level analysis dataset.

The resulting values should be interpreted as aggregate, model-based estimates. Two island ADM2 features are retained with missing under-18 totals because no valid WorldPop raster cells supported those polygons. This project supports further humanitarian assessment; it does not determine aid allocation or identify individual children.
